# Explainable and Externally Validated Machine Learning for Chronic Kidney Disease Classification

**Reproducible analysis pipeline**

This notebook implements the complete experimental pipeline of the study: data preparation, model development, statistical comparison, explainability analysis, feature ablation and external validation. It regenerates all tables (1–7) and figures (2–8) reported in the article.

**Study design at a glance**

| Component | Details |
|---|---|
| Development cohort | UCI Chronic Kidney Disease dataset (ID 336), 400 records, 24 clinical features |
| External cohort | NHANES 2015–2016 and 2017–March 2020 (pre-pandemic), adults ≥ 18 years |
| Outcome (external) | eGFR < 60 mL/min/1.73 m² (CKD-EPI 2021, race-free), KDIGO stages G3a–G5 |
| Machine learning models | Logistic Regression, SVM (RBF), Random Forest, Gradient Boosting, AdaBoost |
| Deep learning models | MLP, ResNet, 1D-CNN, TabNet, FT-Transformer |
| Evaluation | Stratified 80:20 hold-out split, 5-fold stratified cross-validation, 95% CIs (DeLong) |
| Statistical tests | McNemar (continuity-corrected and exact), DeLong AUC test, Bonferroni correction (45 pairs) |
| Explainability | SHAP (Tree, Gradient), TabNet attention masks, LIME, permutation importance, PDP, ICE, PCA decision boundary, consensus ranking with Kendall's W |

**Contents**
1. Environment setup
2. Configuration
3. Statistical utilities
4. Development cohort (UCI)
5. Model definitions
6. Hyperparameter optimisation
7. Hold-out evaluation and computational cost
8. Cross-validation and imputation sensitivity
9. Pairwise statistical comparison
10. Explainability analysis
11. Feature ablation
12. External cohort (NHANES)
13. External validation
14. Figures
15. Reproducibility record

**Running the notebook**: select *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Results are written to Google Drive (`MyDrive/CKD_XAI_results`) and packaged as `CKD_XAI_results.zip`. All random seeds are fixed.

## 1. Environment setup
Installs the libraries not included in the standard Colab image (`pytorch-tabnet`, `shap`, `lime`, `ucimlrepo`, `statsmodels`) and imports all dependencies. Deep learning models run on the GPU when one is available.

In [ ]:
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
_pip("pytorch-tabnet", "shap", "lime", "ucimlrepo", "statsmodels")

from IPython.display import display, Markdown, Image as IPImage
import os, io, json, time, pickle, random, warnings, platform, zipfile, itertools, urllib.request
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import sklearn
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RepeatedStratifiedKFold, RandomizedSearchCV)
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve, confusion_matrix, brier_score_loss)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import loguniform

import torch
import torch.nn as nn
from pytorch_tabnet.tab_model import TabNetClassifier
import shap
import lime.lime_tabular
from statsmodels.stats.contingency_tables import mcnemar

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__, "| sklearn", sklearn.__version__)

## 2. Configuration
Global settings for the experiments. Set `FAST = True` only for a quick functional test; all reported results use `FAST = False`.
If automatic downloads are unavailable, the data can be supplied manually through `UCI_CSV` and `NHANES_LOCAL_DIR`.

| Setting | Value |
|---|---|
| Random seed | 42 |
| Hyperparameter search (ML) | RandomizedSearchCV, 50 iterations, 5-fold inner CV, ROC-AUC |
| Hyperparameter search (DL) | grid: learning rate ∈ {1e-3, 5e-4, 1e-4} × batch size ∈ {16, 32}; TabNet learning rate ∈ {2e-2, 1e-2, 5e-3} |
| DL training | up to 200 epochs, early stopping (patience 15), ReduceLROnPlateau (factor 0.5, patience 7) |
| Permutation importance | 20 repeats, ROC-AUC scoring |

In [ ]:
SEED = 42
FAST = False            # True = quick functional test with reduced iterations

N_ITER_SEARCH   = 8  if FAST else 50     # RandomizedSearchCV iterations
INNER_CV        = 3  if FAST else 5      # inner CV for ML hyperparameter search
DL_TUNE_FOLDS   = 2  if FAST else 3      # inner CV folds for the DL grid search
DL_MAX_EPOCHS   = 15 if FAST else 200
DL_PATIENCE     = 5  if FAST else 15     # early stopping on validation loss
LR_PATIENCE     = 7                      # ReduceLROnPlateau patience
DL_LR_GRID      = [1e-3, 5e-4, 1e-4]
DL_BS_GRID      = [16, 32]
TABNET_LR_GRID  = [2e-2, 1e-2, 5e-3]
PERM_REPEATS    = 5  if FAST else 20     # permutation importance repeats
IMPUTE_REPEATS  = 1  if FAST else 3      # repeats of 5-fold CV for the imputation sensitivity test

# Optional manual data locations (leave None for automatic download)
UCI_CSV = None            # e.g. "/content/kidney_disease.csv" (UCI or Kaggle version)
NHANES_LOCAL_DIR = None   # folder with the NHANES .xpt (or .csv) files if download fails

# Output folder: Google Drive if available, otherwise local
OUT = "/content/CKD_XAI_results"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/CKD_XAI_results"
except Exception as e:
    print("Drive not mounted, saving locally:", e)
if os.environ.get("CKD_OUT"):
    OUT = os.environ["CKD_OUT"]
for sub in ("tables", "figures", "supplementary"):
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)
print("Results folder:", OUT)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

def save_table(df, name, index=False):
    path = os.path.join(OUT, name if "/" in name else os.path.join("tables", name))
    df.to_csv(path, index=index)
    print(f"  saved {name}")
    display(df)
    return path

def save_fig(fig, name, sub=False):
    folder = os.path.join(OUT, "supplementary" if sub else "figures")
    fig.savefig(os.path.join(folder, name), dpi=300, bbox_inches="tight")
    display(fig)                      # show the figure in the notebook as well
    plt.close(fig)
    print(f"  saved {name}")

plt.rcParams.update({"font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
                     "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
LOG = {}   # key results collected during the run

## 3. Statistical utilities

- **DeLong's method** (fast algorithm of Sun and Xu, 2014) for AUC confidence intervals and paired AUC comparison.
- **McNemar's test** on paired predictions, where *b* and *c* are the patients misclassified by only one of the two models:
  $\chi^2 = \dfrac{(|b-c|-1)^2}{b+c}$ (continuity-corrected); the exact binomial version is also reported because *b + c* is small.
- **Decision threshold**: the probability cut-off that maximises F1 on out-of-fold training predictions.
- **Kendall's coefficient of concordance** with tie correction:
  $W = \dfrac{12\sum_j (R_j-\bar R)^2}{m^2(n^3-n) - m\sum T}$, tested with $\chi^2 = m(n-1)W$ on $n-1$ degrees of freedom.

In [ ]:
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x)
    T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N); T2[J] = T
    return T2

def _fast_delong(preds_sorted, m):
    # preds_sorted: (k, n) with the m positives first
    n = preds_sorted.shape[1] - m
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]
    k = preds_sorted.shape[0]
    tx = np.array([_midrank(pos[r]) for r in range(k)])
    ty = np.array([_midrank(neg[r]) for r in range(k)])
    tz = np.array([_midrank(preds_sorted[r]) for r in range(k)])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.atleast_2d(np.cov(v01)); sy = np.atleast_2d(np.cov(v10))
    return aucs, sx / m + sy / n

def _delong_prep(y, *probs):
    y = np.asarray(y).astype(int)
    order = np.argsort(-y, kind="mergesort")      # positives first
    m = int(y.sum())
    P = np.vstack([np.asarray(p, float)[order] for p in probs])
    return P, m

def delong_ci(y, p, alpha=0.95):
    P, m = _delong_prep(y, p)
    auc, cov = _fast_delong(P, m)
    auc = float(auc[0]); se = float(np.sqrt(max(cov[0, 0], 0)))
    z = stats.norm.ppf(1 - (1 - alpha) / 2)
    return auc, max(0.0, auc - z * se), min(1.0, auc + z * se)

def delong_test(y, p1, p2):
    P, m = _delong_prep(y, p1, p2)
    aucs, cov = _fast_delong(P, m)
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    diff = aucs[0] - aucs[1]
    if var <= 1e-15:
        return float(aucs[0]), float(aucs[1]), 0.0, 1.0
    z = diff / np.sqrt(var)
    return float(aucs[0]), float(aucs[1]), float(z), float(2 * stats.norm.sf(abs(z)))

def mcnemar_test(y, pred_a, pred_b):
    ca, cb = (pred_a == y), (pred_b == y)
    b = int(np.sum(ca & ~cb)); c = int(np.sum(~ca & cb))
    table = [[int(np.sum(ca & cb)), b], [c, int(np.sum(~ca & ~cb))]]
    if b + c == 0:
        return b, c, 0.0, 1.0, 1.0
    r_cc = mcnemar(table, exact=False, correction=True)
    r_ex = mcnemar(table, exact=True)
    return b, c, float(r_cc.statistic), float(r_cc.pvalue), float(r_ex.pvalue)

def best_f1_threshold(y, p):
    cands = np.unique(np.r_[np.round(p, 6), 0.5])
    best_t, best_f = 0.5, -1
    for t in cands:
        f = f1_score(y, (p >= t).astype(int), zero_division=0)
        if f > best_f + 1e-12 or (abs(f - best_f) <= 1e-12 and abs(t - 0.5) < abs(best_t - 0.5)):
            best_t, best_f = t, f
    return float(best_t)

def metrics_at(y, p, t):
    yhat = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return dict(Accuracy=accuracy_score(y, yhat), F1=f1_score(y, yhat, zero_division=0),
                Precision=precision_score(y, yhat, zero_division=0),
                Sensitivity=tp / (tp + fn) if tp + fn else np.nan,
                Specificity=tn / (tn + fp) if tn + fp else np.nan,
                TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn))

def kendalls_w(rank_matrix):
    # rank_matrix: (m raters, n items), ranks may contain ties
    R = np.asarray(rank_matrix, float); m, n = R.shape
    Rj = R.sum(axis=0)
    S = np.sum((Rj - Rj.mean()) ** 2)
    T = 0.0
    for r in range(m):
        _, counts = np.unique(R[r], return_counts=True)
        T += np.sum(counts ** 3 - counts)
    W = 12 * S / (m ** 2 * (n ** 3 - n) - m * T)
    chi2 = m * (n - 1) * W
    p = stats.chi2.sf(chi2, n - 1)
    return float(W), float(chi2), float(p)

print("helpers ready")

## 4. Development cohort: UCI CKD dataset
The UCI dataset (ID 336) is loaded with `ucimlrepo`. Cleaning removes stray whitespace and tab characters, converts `?` to missing, encodes the ten binary categorical features with a fixed mapping (so no encoder is fitted on test data), and maps the label to CKD = 1, not-CKD = 0.

A stratified 80:20 split (320 training / 80 test records) is created once and kept fixed for all hold-out analyses.

In [ ]:
RENAME = {"wbcc": "wc", "rbcc": "rc", "classification": "class"}
NUMERIC = ["age", "bp", "sg", "al", "su", "bgr", "bu", "sc", "sod", "pot", "hemo", "pcv", "wc", "rc"]
BINARY_MAP = {
    "rbc": {"normal": 0, "abnormal": 1}, "pc": {"normal": 0, "abnormal": 1},
    "pcc": {"notpresent": 0, "present": 1}, "ba": {"notpresent": 0, "present": 1},
    "htn": {"no": 0, "yes": 1}, "dm": {"no": 0, "yes": 1}, "cad": {"no": 0, "yes": 1},
    "appet": {"good": 0, "poor": 1}, "pe": {"no": 0, "yes": 1}, "ane": {"no": 0, "yes": 1},
}
FEATURES = ["age", "bp", "sg", "al", "su", "rbc", "pc", "pcc", "ba", "bgr", "bu", "sc", "sod",
            "pot", "hemo", "pcv", "wc", "rc", "htn", "dm", "cad", "appet", "pe", "ane"]

def load_uci_raw():
    if UCI_CSV:
        return pd.read_csv(UCI_CSV)
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=336)
        return pd.concat([ds.data.features, ds.data.targets], axis=1)
    except Exception as e:
        print("ucimlrepo failed:", e)
        return pd.read_csv("https://archive.ics.uci.edu/static/public/336/data.csv")

def clean_uci(raw):
    df = raw.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.rename(columns=RENAME).drop(columns=["id"], errors="ignore")
    for c in df.columns:
        if df[c].dtype == object or str(df[c].dtype).startswith("str"):
            df[c] = df[c].astype(str).str.strip().str.replace("\t", "", regex=False)
            df[c] = df[c].replace({"?": np.nan, "nan": np.nan, "": np.nan, "None": np.nan})
    for c in NUMERIC:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c, mp in BINARY_MAP.items():
        df[c] = df[c].map(lambda v: mp.get(str(v).strip().lower(), np.nan) if pd.notna(v) else np.nan)
    df["class"] = df["class"].astype(str).str.strip().str.lower().map({"ckd": 1, "notckd": 0})
    df = df[df["class"].notna()].reset_index(drop=True)
    return df[FEATURES].astype(float), df["class"].astype(int).values

uci_raw = load_uci_raw()
X_all, y_all = clean_uci(uci_raw)
n_ckd, n_not = int(y_all.sum()), int((1 - y_all).sum())
miss = X_all.isna().mean().sort_values(ascending=False)
print(f"UCI: {X_all.shape[0]} records, {X_all.shape[1]} features | CKD {n_ckd} / notCKD {n_not}")
print(f"Overall missing cells: {int(X_all.isna().sum().sum())}; max per-feature missing: "
      f"{miss.iloc[0]*100:.1f}% ({miss.index[0]})")
LOG["uci"] = dict(n=int(len(y_all)), ckd=n_ckd, notckd=n_not, ratio=round(n_ckd / n_not, 2),
                  max_missing_pct=round(float(miss.iloc[0] * 100), 1), max_missing_feature=miss.index[0])
save_table(miss.rename("missing_fraction").reset_index().rename(columns={"index": "feature"}),
           "supplementary/S_uci_missingness.csv")

# Held-out split
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED)
X_tr, X_te = X_tr.reset_index(drop=True), X_te.reset_index(drop=True)
print(f"Train {len(y_tr)} (CKD {y_tr.sum()}) | Test {len(y_te)} (CKD {y_te.sum()})")
LOG["split"] = dict(train=int(len(y_tr)), test=int(len(y_te)), test_ckd=int(y_te.sum()))

## 5. Model definitions
Every model is wrapped in a pipeline **KNN imputation (k = 5) → standard scaling → classifier**, so imputation and scaling are always fitted on training data only. Class imbalance is handled with balanced class weights (ML) or a weighted binary cross-entropy loss (DL).

| Model | Architecture |
|---|---|
| MLP | 256-128-64-1, batch normalisation, 30% dropout |
| ResNet | input projection to 128 units, two residual blocks, batch normalisation |
| 1D-CNN | two convolutional layers (64, 128 filters, kernel 3), global average pooling; features ordered by clinical group (haematological, biochemical, urinalysis, demographic, comorbidity) |
| TabNet | sequential attention with sparse feature selection (n_d = n_a = 8, 5 steps, γ = 1.3) |
| FT-Transformer | each feature embedded as a token (d = 16) with a [CLS] token, 2 Transformer layers, 2 attention heads, feed-forward size 64 |

In [ ]:
GROUP_ORDER = ["hemo", "pcv", "rc", "wc",                      # haematological
               "bgr", "bu", "sc", "sod", "pot",                # biochemical
               "sg", "al", "su", "rbc", "pc", "pcc", "ba",     # urinalysis
               "age", "bp",                                    # demographic / vital
               "htn", "dm", "cad", "appet", "pe", "ane"]       # comorbidity / symptoms

def cnn_order(feature_names):
    # 1D-CNN: features arranged by clinical modality group
    return tuple(sorted(range(len(feature_names)), key=lambda i: GROUP_ORDER.index(feature_names[i])))

class MLPNet(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

class ResBlock(nn.Module):
    def __init__(self, h, p=0.2):
        super().__init__()
        self.f = nn.Sequential(nn.BatchNorm1d(h), nn.Linear(h, h), nn.ReLU(), nn.Dropout(p), nn.Linear(h, h))
    def forward(self, x): return x + self.f(x)

class ResNetNet(nn.Module):
    def __init__(self, d, h=128):
        super().__init__()
        self.inp = nn.Linear(d, h)
        self.blocks = nn.Sequential(ResBlock(h), ResBlock(h))
        self.out = nn.Sequential(nn.BatchNorm1d(h), nn.ReLU(), nn.Linear(h, 1))
    def forward(self, x): return self.out(self.blocks(self.inp(x))).squeeze(-1)

class CNN1DNet(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(128, 1))
    def forward(self, x):
        h = self.conv(x.unsqueeze(1)).mean(dim=-1)       # global average pooling
        return self.head(h).squeeze(-1)

class FTTransformerNet(nn.Module):
    # Feature Tokenizer + Transformer (Gorishniy et al., 2021): each feature becomes a
    # learnable embedding token; a [CLS] token is prepended; attention runs over feature tokens.
    def __init__(self, d, d_token=16, n_heads=2, n_layers=2, d_ffn=64, dropout=0.1):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d, d_token) * 0.02)
        self.b = nn.Parameter(torch.zeros(d, d_token))
        self.cls = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)
        layer = nn.TransformerEncoderLayer(d_token, n_heads, d_ffn, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_token), nn.ReLU(), nn.Linear(d_token, 1))
    def forward(self, x):
        tok = x.unsqueeze(-1) * self.W + self.b                 # (B, d, d_token)
        tok = torch.cat([self.cls.expand(x.shape[0], -1, -1), tok], dim=1)
        return self.head(self.encoder(tok)[:, 0]).squeeze(-1)

ARCHS = {"mlp": MLPNet, "resnet": ResNetNet, "cnn": CNN1DNet, "ftt": FTTransformerNet}

class TorchTabular(BaseEstimator, ClassifierMixin):
    # sklearn-compatible wrapper: Adam, class-weighted BCE, 15% validation split,
    # early stopping on validation loss, ReduceLROnPlateau(factor 0.5)
    def __init__(self, arch="mlp", lr=1e-3, batch_size=32, max_epochs=200, patience=15,
                 lr_patience=7, feature_order=None, seed=42):
        self.arch = arch; self.lr = lr; self.batch_size = batch_size; self.max_epochs = max_epochs
        self.patience = patience; self.lr_patience = lr_patience
        self.feature_order = feature_order; self.seed = seed

    def _prep(self, X):
        X = np.asarray(X, dtype=np.float32)
        if self.feature_order is not None:
            X = X[:, list(self.feature_order)]
        return X

    def fit(self, X, y):
        set_seed(self.seed)
        X = self._prep(X); y = np.asarray(y).astype(np.float32)
        Xt, Xv, yt, yv = train_test_split(X, y, test_size=0.15, stratify=y, random_state=self.seed)
        model = ARCHS[self.arch](X.shape[1]).to(DEVICE)
        pos_w = torch.tensor((len(yt) - yt.sum()) / max(yt.sum(), 1), device=DEVICE)
        lossf = nn.BCEWithLogitsLoss(pos_weight=pos_w)
        opt = torch.optim.Adam(model.parameters(), lr=self.lr)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=self.lr_patience)
        Xt_t, yt_t = torch.tensor(Xt, device=DEVICE), torch.tensor(yt, device=DEVICE)
        Xv_t, yv_t = torch.tensor(Xv, device=DEVICE), torch.tensor(yv, device=DEVICE)
        g = torch.Generator().manual_seed(self.seed)
        best, best_state, wait = np.inf, None, 0
        for epoch in range(self.max_epochs):
            model.train()
            perm = torch.randperm(len(Xt_t), generator=g).to(DEVICE)
            for i in range(0, len(perm), self.batch_size):
                idx = perm[i:i + self.batch_size]
                if len(idx) < 2:          # BatchNorm needs >1 sample
                    continue
                opt.zero_grad()
                loss = lossf(model(Xt_t[idx]), yt_t[idx])
                loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                vl = lossf(model(Xv_t), yv_t).item()
            sched.step(vl)
            if vl < best - 1e-5:
                best, wait = vl, 0
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
                if wait >= self.patience:
                    break
        model.load_state_dict(best_state)
        model.eval()
        self.model_, self.n_epochs_, self.classes_ = model, epoch + 1, np.array([0, 1])
        return self

    def decision_logits(self, X):
        X = torch.tensor(self._prep(X), device=DEVICE)
        with torch.no_grad():
            return self.model_(X).cpu().numpy()

    def predict_proba(self, X):
        p = 1 / (1 + np.exp(-self.decision_logits(X)))
        return np.c_[1 - p, p]

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

class TabNetWrap(BaseEstimator, ClassifierMixin):
    def __init__(self, n_d=8, n_a=8, n_steps=5, gamma=1.3, lr=2e-2, batch_size=256,
                 virtual_batch_size=128, max_epochs=200, patience=15, seed=42):
        self.n_d = n_d; self.n_a = n_a; self.n_steps = n_steps; self.gamma = gamma; self.lr = lr
        self.batch_size = batch_size; self.virtual_batch_size = virtual_batch_size
        self.max_epochs = max_epochs; self.patience = patience; self.seed = seed

    def fit(self, X, y):
        set_seed(self.seed)
        X = np.asarray(X, np.float32); y = np.asarray(y).astype(int)
        Xt, Xv, yt, yv = train_test_split(X, y, test_size=0.15, stratify=y, random_state=self.seed)
        bs = min(self.batch_size, len(Xt)); vbs = min(self.virtual_batch_size, bs)
        self.model_ = TabNetClassifier(n_d=self.n_d, n_a=self.n_a, n_steps=self.n_steps, gamma=self.gamma,
                                       optimizer_params=dict(lr=self.lr), seed=self.seed, verbose=0,
                                       device_name=DEVICE)
        import contextlib
        with contextlib.redirect_stdout(io.StringIO()):       # silence TabNet's per-run messages
            self.model_.fit(Xt, yt, eval_set=[(Xv, yv)], eval_metric=["logloss"], max_epochs=self.max_epochs,
                            patience=self.patience, batch_size=bs, virtual_batch_size=vbs,
                            weights=1, drop_last=True)
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(np.asarray(X, np.float32))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

ML_NAMES = ["Logistic Regression", "SVM (RBF)", "Random Forest", "Gradient Boosting", "AdaBoost"]
DL_NAMES = ["MLP", "ResNet", "1D-CNN", "TabNet", "FT-Transformer"]
ALL_NAMES = ML_NAMES + DL_NAMES
SHORT = {"Logistic Regression": "LR", "SVM (RBF)": "SVM", "Random Forest": "RF", "Gradient Boosting": "GB",
         "AdaBoost": "Ada", "MLP": "MLP", "ResNet": "ResNet", "1D-CNN": "1D-CNN", "TabNet": "TabNet",
         "FT-Transformer": "FT-T"}
DL_ARCH = {"MLP": "mlp", "ResNet": "resnet", "1D-CNN": "cnn", "FT-Transformer": "ftt"}
SAMPLE_WEIGHT_MODELS = {"Gradient Boosting", "AdaBoost"}   # class weighting via sample weights

def base_estimator(name, feature_names):
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED)
    if name == "SVM (RBF)":
        return SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED)
    if name == "Random Forest":
        return RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1)
    if name == "Gradient Boosting":
        return GradientBoostingClassifier(random_state=SEED)
    if name == "AdaBoost":
        return AdaBoostClassifier(random_state=SEED)
    if name == "TabNet":
        return TabNetWrap(max_epochs=DL_MAX_EPOCHS, patience=DL_PATIENCE, seed=SEED)
    order = cnn_order(feature_names) if name == "1D-CNN" else None
    return TorchTabular(arch=DL_ARCH[name], max_epochs=DL_MAX_EPOCHS, patience=DL_PATIENCE,
                        lr_patience=LR_PATIENCE, feature_order=order, seed=SEED)

SEARCH_SPACE = {
    "Logistic Regression": {"clf__C": loguniform(1e-3, 1e2)},
    "SVM (RBF)": {"clf__C": loguniform(1e-2, 1e3), "clf__gamma": loguniform(1e-4, 1e0)},
    "Random Forest": {"clf__n_estimators": [100, 200, 300, 500], "clf__max_depth": [None, 5, 10, 20],
                      "clf__min_samples_leaf": [1, 2, 4], "clf__max_features": ["sqrt", "log2", None]},
    "Gradient Boosting": {"clf__n_estimators": [50, 100, 200, 300], "clf__learning_rate": loguniform(1e-2, 3e-1),
                          "clf__max_depth": [2, 3, 4, 5], "clf__subsample": [0.7, 0.85, 1.0]},
    "AdaBoost": {"clf__n_estimators": [50, 100, 150, 200, 300], "clf__learning_rate": loguniform(1e-2, 1.5)},
}

def make_pipe(name, feature_names, params=None, imputer=None):
    pipe = Pipeline([("imp", imputer if imputer is not None else KNNImputer(n_neighbors=5)),
                     ("sc", StandardScaler()),
                     ("clf", base_estimator(name, list(feature_names)))])
    if params:
        pipe.set_params(**params)
    return pipe

def fit_pipe(pipe, name, X, y):
    if name in SAMPLE_WEIGHT_MODELS:
        pipe.fit(X, y, clf__sample_weight=compute_sample_weight("balanced", y))
    else:
        pipe.fit(X, y)
    return pipe

def proba(pipe, X):
    return pipe.predict_proba(X)[:, 1]

print("models ready:", ALL_NAMES)

## 6. Hyperparameter optimisation
Hyperparameters are selected on the **training set only**. ML models use randomised search; DL models use a grid search scored by inner cross-validated ROC-AUC. The selected values are saved to `supplementary/S_hyperparameters.csv`.

In [ ]:
BEST_PARAMS, TUNING_ROWS = {}, []
inner = StratifiedKFold(INNER_CV, shuffle=True, random_state=SEED)

for name in ML_NAMES:
    t0 = time.time()
    search = RandomizedSearchCV(make_pipe(name, FEATURES), SEARCH_SPACE[name], n_iter=N_ITER_SEARCH,
                                scoring="roc_auc", cv=inner, random_state=SEED, n_jobs=-1, refit=False)
    fit_kw = {"clf__sample_weight": compute_sample_weight("balanced", y_tr)} if name in SAMPLE_WEIGHT_MODELS else {}
    search.fit(X_tr, y_tr, **fit_kw)
    BEST_PARAMS[name] = search.best_params_
    TUNING_ROWS.append(dict(model=name, best_params=json.dumps({k.replace("clf__", ""): (round(v, 6) if isinstance(v, float) else v)
                                                                for k, v in search.best_params_.items()}),
                            inner_cv_auc=round(search.best_score_, 4), seconds=round(time.time() - t0, 1)))
    print(f"  {name:<20} inner AUC={search.best_score_:.4f}  {search.best_params_}")

def inner_cv_auc(name, params, X, y, folds):
    skf = StratifiedKFold(folds, shuffle=True, random_state=SEED)
    aucs = []
    for tr, va in skf.split(X, y):
        pipe = fit_pipe(make_pipe(name, FEATURES, params), name, X.iloc[tr], y[tr])
        aucs.append(roc_auc_score(y[va], proba(pipe, X.iloc[va])))
    return float(np.mean(aucs))

for name in DL_NAMES:
    t0 = time.time()
    if name == "TabNet":
        grid = [{"clf__lr": lr} for lr in TABNET_LR_GRID]
    else:
        grid = [{"clf__lr": lr, "clf__batch_size": bs} for lr in DL_LR_GRID for bs in DL_BS_GRID]
    scores = [(inner_cv_auc(name, g, X_tr, y_tr, DL_TUNE_FOLDS), g) for g in grid]
    best_auc, best_g = max(scores, key=lambda s: s[0])
    BEST_PARAMS[name] = best_g
    TUNING_ROWS.append(dict(model=name, best_params=json.dumps({k.replace("clf__", ""): v for k, v in best_g.items()}),
                            inner_cv_auc=round(best_auc, 4), seconds=round(time.time() - t0, 1)))
    print(f"  {name:<20} inner AUC={best_auc:.4f}  {best_g}")

save_table(pd.DataFrame(TUNING_ROWS), "supplementary/S_hyperparameters.csv")

## 7. Hold-out evaluation and computational cost
Each tuned model is trained on the 320 training records and evaluated on the 80 held-out records. The decision threshold is fixed beforehand from out-of-fold training predictions. Reported: accuracy, F1, precision, sensitivity, specificity, ROC-AUC with 95% DeLong CI, training time, per-patient inference latency and serialised model size (**Table 3**).

In [ ]:
def oof_proba(name, params, X, y, feature_names, folds=5):
    skf = StratifiedKFold(folds, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y))
    for tr, va in skf.split(X, y):
        pipe = fit_pipe(make_pipe(name, feature_names, params), name, X.iloc[tr], y[tr])
        oof[va] = proba(pipe, X.iloc[va])
    return oof

def model_size_kb(pipe):
    try:
        return len(pickle.dumps(pipe)) / 1024
    except Exception:
        clf = pipe.named_steps["clf"]
        if isinstance(clf, TabNetWrap):
            import tempfile
            path = clf.model_.save_model(os.path.join(tempfile.mkdtemp(), "tabnet"))
            return os.path.getsize(path) / 1024
        buf = io.BytesIO(); torch.save(clf.model_.state_dict(), buf)
        return buf.tell() / 1024

HELD, THRESH, FITTED, COMPLEX = {}, {}, {}, []
for name in ALL_NAMES:
    params = BEST_PARAMS[name]
    THRESH[name] = best_f1_threshold(y_tr, oof_proba(name, params, X_tr, y_tr, FEATURES))
    pipe = make_pipe(name, FEATURES, params)
    t0 = time.perf_counter(); fit_pipe(pipe, name, X_tr, y_tr); train_s = time.perf_counter() - t0
    reps = 20
    t0 = time.perf_counter()
    for _ in range(reps):
        p = proba(pipe, X_te)
    infer_ms = (time.perf_counter() - t0) / reps / len(X_te) * 1000
    FITTED[name] = pipe
    auc, lo, hi = delong_ci(y_te, p)
    m = metrics_at(y_te, p, THRESH[name])
    HELD[name] = dict(prob=p, pred=(p >= THRESH[name]).astype(int), auc=auc, ci=(lo, hi), **m)
    COMPLEX.append(dict(Model=name, Type="ML" if name in ML_NAMES else "DL",
                        **{"Training time (s)": round(train_s, 3),
                           "Inference latency (ms/patient)": round(infer_ms, 4),
                           "Model size (KB)": round(model_size_kb(pipe), 1)}))
    print(f"  {name:<20} thr={THRESH[name]:.3f} Acc={m['Accuracy']:.4f} F1={m['F1']:.4f} "
          f"Sens={m['Sensitivity']:.3f} Spec={m['Specificity']:.3f} AUC={auc:.4f} [{lo:.3f}-{hi:.3f}]")

table3 = pd.DataFrame(COMPLEX)
save_table(table3, "Table3_complexity.csv")

## 8. Five-fold cross-validation
All ten models are re-evaluated with stratified 5-fold cross-validation on the full development cohort, using the tuned hyperparameters. Hold-out and cross-validated results are combined in **Table 2**.

In [ ]:
CV_SPLITS = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_all, y_all))
CV = {}
for name in ALL_NAMES:
    aucs, oof = [], np.zeros(len(y_all))
    for tr, va in CV_SPLITS:
        pipe = fit_pipe(make_pipe(name, FEATURES, BEST_PARAMS[name]), name, X_all.iloc[tr], y_all[tr])
        oof[va] = proba(pipe, X_all.iloc[va])
        aucs.append(roc_auc_score(y_all[va], oof[va]))
    CV[name] = dict(auc_mean=float(np.mean(aucs)), auc_sd=float(np.std(aucs, ddof=1)), fold_aucs=aucs, oof=oof)
    print(f"  {name:<20} CV-AUC = {np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}")

rows = []
for name in ALL_NAMES:
    h = HELD[name]
    rows.append({"Model": name, "Type": "ML" if name in ML_NAMES else "DL",
                 "Accuracy (%)": round(100 * h["Accuracy"], 2), "F1": round(h["F1"], 3),
                 "Sensitivity": round(h["Sensitivity"], 3), "Specificity": round(h["Specificity"], 3),
                 "ROC-AUC": round(h["auc"], 3), "ROC-AUC 95% CI": f"{h['ci'][0]:.3f}-{h['ci'][1]:.3f}",
                 "TP": h["TP"], "TN": h["TN"], "FP": h["FP"], "FN": h["FN"], "Threshold": round(THRESH[name], 3),
                 "CV-AUC (5-fold)": f"{CV[name]['auc_mean']:.3f} ± {CV[name]['auc_sd']:.3f}"})
table2 = pd.DataFrame(rows)
save_table(table2, "Table2_performance.csv")
display(table2) if "display" in globals() else print(table2.to_string())

### 8.1 Imputation sensitivity
Random Forest performance is compared across KNN imputation (k = 3, 5, 7) and multiple imputation by chained equations (MICE), using repeated stratified 5-fold cross-validation and paired Wilcoxon signed-rank tests against KNN (k = 5).

In [ ]:
imputers = {"KNN (k=3)": KNNImputer(n_neighbors=3), "KNN (k=5)": KNNImputer(n_neighbors=5),
            "KNN (k=7)": KNNImputer(n_neighbors=7),
            "MICE": IterativeImputer(max_iter=10, random_state=SEED)}
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=IMPUTE_REPEATS, random_state=SEED)
imp_scores = {k: [] for k in imputers}
for tr, va in rskf.split(X_all, y_all):
    for k, imp in imputers.items():
        pipe = make_pipe("Random Forest", FEATURES, BEST_PARAMS["Random Forest"], imputer=clone(imp))
        fit_pipe(pipe, "Random Forest", X_all.iloc[tr], y_all[tr])
        imp_scores[k].append(roc_auc_score(y_all[va], proba(pipe, X_all.iloc[va])))
imp_rows = []
for k, v in imp_scores.items():
    if k == "KNN (k=5)":
        p = np.nan
    else:
        diff = np.array(v) - np.array(imp_scores["KNN (k=5)"])
        p = 1.0 if np.allclose(diff, 0) else stats.wilcoxon(v, imp_scores["KNN (k=5)"]).pvalue
    imp_rows.append(dict(Imputation=k, CV_AUC_mean=round(np.mean(v), 4), CV_AUC_sd=round(np.std(v, ddof=1), 4),
                         n_folds=len(v), Wilcoxon_p_vs_KNN5=p))
save_table(pd.DataFrame(imp_rows), "supplementary/S_imputation_sensitivity.csv")


## 9. Pairwise statistical comparison
All 45 model pairs are compared on the hold-out test set with McNemar's test (classification decisions) and DeLong's test (ROC-AUC). Significance is assessed at the Bonferroni-corrected level α = 0.05 / 45 = 0.0011 (**Table 4**).

In [ ]:
ALPHA_BONF = 0.05 / 45
rows = []
for a, b in itertools.combinations(ALL_NAMES, 2):
    bb, cc, chi2, p_cc, p_ex = mcnemar_test(y_te, HELD[a]["pred"], HELD[b]["pred"])
    auc_a, auc_b, z, p_dl = delong_test(y_te, HELD[a]["prob"], HELD[b]["prob"])
    rows.append({"Model A": a, "Model B": b, "b (A right, B wrong)": bb, "c (A wrong, B right)": cc,
                 "McNemar chi2 (cc)": round(chi2, 3), "McNemar p (cc)": p_cc, "McNemar p (exact)": p_ex,
                 "McNemar significant (Bonferroni)": p_cc < ALPHA_BONF,
                 "AUC A": round(auc_a, 4), "AUC B": round(auc_b, 4), "DeLong z": round(z, 3),
                 "DeLong p": p_dl, "DeLong significant (Bonferroni)": p_dl < ALPHA_BONF})
table4 = pd.DataFrame(rows)
save_table(table4, "Table4_pairwise_tests.csv")
print(f"Bonferroni threshold = {ALPHA_BONF:.5f}")
print("Significant McNemar pairs:", int(table4["McNemar significant (Bonferroni)"].sum()), "/ 45")
print("Significant DeLong pairs :", int(table4["DeLong significant (Bonferroni)"].sum()), "/ 45")
print(table4[["Model A", "Model B", "McNemar chi2 (cc)", "McNemar p (cc)", "DeLong p"]]
      .sort_values("McNemar p (cc)").head(12).to_string(index=False))

## 10. Explainability analysis
Ten complementary explanation methods are applied:

1. SHAP TreeExplainer (Random Forest), exact Shapley values
2. SHAP GradientExplainer (MLP)
3. TabNet attention masks (built-in instance-wise feature selection)
4–5. LIME local surrogates (Random Forest and MLP)
6. Permutation importance for all ML models (ROC-AUC scoring)
7. Partial dependence plots (PDP)
8. Individual conditional expectation (ICE) curves
9. Two-dimensional PCA decision boundary
10. Cross-method consensus ranking: the mean rank across SHAP-RF, SHAP-MLP, permutation importance and Gini importance, with agreement measured by Kendall's W (**Table 7**)

The Shapley value of feature *j* for patient *x* is
$\phi_j = \sum_{S \subseteq F \setminus \{j\}} \dfrac{|S|!\,(|F|-|S|-1)!}{|F|!}\,[f_{S\cup\{j\}}(x) - f_S(x)]$.

In [ ]:
rf_pipe = FITTED["Random Forest"]
pre = Pipeline(rf_pipe.steps[:-1])                    # fitted imputer + scaler
Xtr_s = pre.transform(X_tr); Xte_s = pre.transform(X_te)

# (1) SHAP - Random Forest
expl_rf = shap.TreeExplainer(rf_pipe.named_steps["clf"])
sv = expl_rf.shap_values(Xte_s)
if isinstance(sv, list):
    sv_rf = np.asarray(sv[1])
else:
    sv_rf = np.asarray(sv)
    if sv_rf.ndim == 3:
        sv_rf = sv_rf[:, :, 1]
shap_rf = np.abs(sv_rf).mean(axis=0)

# (2) SHAP - MLP (GradientExplainer on the logit output)
mlp_pipe = FITTED["MLP"]
mlp_net = mlp_pipe.named_steps["clf"].model_
Xtr_m = torch.tensor(Pipeline(mlp_pipe.steps[:-1]).transform(X_tr), dtype=torch.float32, device=DEVICE)
Xte_m = torch.tensor(Pipeline(mlp_pipe.steps[:-1]).transform(X_te), dtype=torch.float32, device=DEVICE)
class _Logit(nn.Module):
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, x): return self.net(x).unsqueeze(-1)
mlp_net.eval()
expl_mlp = shap.GradientExplainer(_Logit(mlp_net), Xtr_m[:min(100, len(Xtr_m))])
sv_m = expl_mlp.shap_values(Xte_m)
sv_m = np.asarray(sv_m[0] if isinstance(sv_m, list) else sv_m)
if sv_m.ndim == 3:
    sv_m = sv_m[..., 0]
shap_mlp = np.abs(sv_m).mean(axis=0)

# (3) TabNet attention masks (built-in instance-wise feature selection)
tab_pipe = FITTED["TabNet"]
M_explain, _ = tab_pipe.named_steps["clf"].model_.explain(
    Pipeline(tab_pipe.steps[:-1]).transform(X_te).astype(np.float32))
tabnet_imp = M_explain.sum(axis=0) / M_explain.sum()

# (6) permutation importance (test set, ROC-AUC) for all ML models
PERM = {}
for name in ML_NAMES:
    r = permutation_importance(FITTED[name], X_te, y_te, n_repeats=PERM_REPEATS, scoring="roc_auc",
                               random_state=SEED, n_jobs=-1)
    PERM[name] = r.importances_mean
perm_rf = PERM["Random Forest"]
gini_rf = rf_pipe.named_steps["clf"].feature_importances_

# (10) consensus ranking across four attribution methods
def to_rank(v):
    return stats.rankdata(-np.asarray(v), method="average")
rank_df = pd.DataFrame({"SHAP-RF": to_rank(shap_rf), "SHAP-MLP": to_rank(shap_mlp),
                        "Permutation": to_rank(perm_rf), "Gini (RF)": to_rank(gini_rf)}, index=FEATURES)
rank_df["Consensus (mean rank)"] = rank_df.mean(axis=1)
rank_df["TabNet mask (rank, not in consensus)"] = to_rank(tabnet_imp)
rank_df = rank_df.sort_values("Consensus (mean rank)")
CONSENSUS = list(rank_df.index)

W_all, chi_all, p_all = kendalls_w(rank_df[["SHAP-RF", "SHAP-MLP", "Permutation", "Gini (RF)"]].T.values)
top10 = rank_df.iloc[:10][["SHAP-RF", "SHAP-MLP", "Permutation", "Gini (RF)"]]
W_10, chi_10, p_10 = kendalls_w(np.vstack([stats.rankdata(top10[c]) for c in top10.columns]))
print(f"Kendall's W (all 24 features) = {W_all:.3f}, chi2 = {chi_all:.2f}, p = {p_all:.2e}")
print(f"Kendall's W (top-10 consensus features) = {W_10:.3f}, p = {p_10:.3g}")
LOG["kendall"] = dict(W_all=round(W_all, 3), p_all=p_all, W_top10=round(W_10, 3), p_top10=p_10)
LOG["consensus_top5"] = CONSENSUS[:5]

CLINICAL = {"hemo": "Renal anaemia (reduced erythropoietin)", "pcv": "Packed cell volume, anaemia",
            "rc": "Red cell count, impaired erythropoiesis", "sc": "Serum creatinine, GFR surrogate",
            "al": "Albuminuria, glomerular damage", "sg": "Urine specific gravity, concentrating ability",
            "bu": "Blood urea, reduced clearance", "bgr": "Blood glucose, diabetic nephropathy",
            "htn": "Hypertension, cause/accelerator", "dm": "Diabetes, leading cause of CKD",
            "sod": "Sodium, advanced CKD", "pot": "Potassium, late-stage CKD", "rbc": "Urine red cells",
            "pc": "Urine pus cells", "su": "Urine sugar", "bp": "Diastolic blood pressure", "age": "Age",
            "wc": "White cell count", "appet": "Appetite (uraemia)", "pe": "Pedal oedema",
            "ane": "Anaemia (clinical flag)", "pcc": "Pus cell clumps", "ba": "Bacteria", "cad": "Coronary artery disease"}
table7 = rank_df.copy()
table7.insert(0, "Feature", table7.index)
table7["Clinical significance"] = [CLINICAL.get(f, "") for f in table7.index]
save_table(table7.round(2), "Table7_xai_consensus.csv")
with open(os.path.join(OUT, "tables", "Table7_kendall_W.txt"), "w") as f:
    f.write(f"Kendall's W (4 methods x 24 features) = {W_all:.4f}, chi2 = {chi_all:.3f}, df = 23, p = {p_all:.3e}\n")
    f.write(f"Kendall's W (top-10 consensus features) = {W_10:.4f}, p = {p_10:.3e}\n")


## 11. Feature ablation
Features are added to the Random Forest in the order of the consensus ranking from Section 10 (top 3, 5, 8, 12 and all 24), and each subset is evaluated with the same 5-fold splits (**Table 6**). This tests how much of the predictive information is retained by small, low-cost feature sets.

In [ ]:
ABL_SIZES = [3, 5, 8, 12, len(FEATURES)]
abl_rows, ABL = [], {}
for k in ABL_SIZES:
    feats = CONSENSUS[:k]
    aucs, oof = [], np.zeros(len(y_all))
    for tr, va in CV_SPLITS:
        pipe = fit_pipe(make_pipe("Random Forest", feats, BEST_PARAMS["Random Forest"]), "Random Forest",
                        X_all.iloc[tr][feats], y_all[tr])
        oof[va] = proba(pipe, X_all.iloc[va][feats])
        aucs.append(roc_auc_score(y_all[va], oof[va]))
    m = metrics_at(y_all, oof, 0.5)
    label = f"Top-{k}" if k < len(FEATURES) else f"All-{k}"
    ABL[label] = dict(auc=np.mean(aucs), sd=np.std(aucs, ddof=1), sens=m["Sensitivity"], spec=m["Specificity"])
    abl_rows.append({"Feature set": label, "Features included": ", ".join(feats) if k <= 12 else "all",
                     "CV-AUC (mean ± SD)": f"{np.mean(aucs):.3f} ± {np.std(aucs, ddof=1):.3f}",
                     "Sensitivity": round(m["Sensitivity"], 3), "Specificity": round(m["Specificity"], 3),
                     "Accuracy": round(m["Accuracy"], 3)})
table6 = pd.DataFrame(abl_rows)
save_table(table6, "Table6_ablation.csv")


## 12. External cohort: NHANES
Data files are downloaded from the CDC NHANES repository for the 2015–2016 (`_I`) and 2017–March 2020 pre-pandemic (`P_`) cycles. The 14 features shared with the UCI dataset are harmonised to UCI units:

| Feature | NHANES source | Harmonisation |
|---|---|---|
| hemo, pcv, rc, wc | CBC: LBXHGB, LBXHCT, LBXRBCSI, LBXWBCSI | pcv = haematocrit; wc × 1000 (cells/cmm) |
| bgr, bu, sc, sod, pot | BIOPRO: LBXSGL, LBXSBU, LBXSCR, LBXSNASI, LBXSKSI | urea = BUN × 2.14 |
| al | ALB_CR: URXUMA (mg/L) | mapped to dipstick grade 0–4 |
| bp | BPX / BPXO | mean diastolic pressure of valid readings |
| htn, dm, cad | BPQ020, DIQ010, MCQ160C | yes = 1; no or borderline = 0 |

CKD is defined as eGFR < 60 mL/min/1.73 m² using the race-free CKD-EPI 2021 equation. Adults (≥ 18 years) with complete data on all 14 features are retained. **Table 1** summarises both cohorts, and a feature-shift table compares their distributions.

In [ ]:
EXT_FEATURES = ["hemo", "pcv", "rc", "wc", "bgr", "bu", "sc", "sod", "pot", "al", "bp", "htn", "dm", "cad"]
CYCLES = {
    "2015-2016": dict(year=2015, files=dict(DEMO="DEMO_I", CBC="CBC_I", BIO="BIOPRO_I", ALB="ALB_CR_I",
                                            BP="BPX_I", BPQ="BPQ_I", DIQ="DIQ_I", MCQ="MCQ_I")),
    "2017-2020": dict(year=2017, files=dict(DEMO="P_DEMO", CBC="P_CBC", BIO="P_BIOPRO", ALB="P_ALB_CR",
                                            BP="P_BPXO", BPQ="P_BPQ", DIQ="P_DIQ", MCQ="P_MCQ")),
}
NH_DIR = NHANES_LOCAL_DIR or os.path.join("/content", "nhanes_raw")
os.makedirs(NH_DIR, exist_ok=True)

def read_nhanes(fname, period, year):
    for ext in (".xpt", ".XPT", ".csv"):
        p = os.path.join(NH_DIR, fname + ext)
        if os.path.exists(p):
            return pd.read_csv(p) if ext == ".csv" else pd.read_sas(p, format="xport", encoding="utf-8")
    urls = [f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{fname}.xpt",
            f"https://wwwn.cdc.gov/Nchs/Nhanes/{period}/{fname}.XPT"]
    for u in urls:
        try:
            req = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0"})
            data = urllib.request.urlopen(req, timeout=120).read()
            if data[:6] == b"HEADER":                      # genuine SAS transport file
                p = os.path.join(NH_DIR, fname + ".xpt")
                open(p, "wb").write(data)
                return pd.read_sas(p, format="xport", encoding="utf-8")
        except Exception as e:
            print(f"    {u}: {e}")
    raise RuntimeError(f"Could not obtain {fname}. Download it from the CDC NHANES website, "
                       f"upload it to a folder and set NHANES_LOCAL_DIR.")

def albumin_grade(mg_per_L):
    # approximate urine dipstick grade (UCI 'al', 0-5) from urine albumin concentration
    bins = [-np.inf, 300, 1000, 3000, 10000, np.inf]          # mg/L: <30 mg/dL -> 0, 30 -> 1, 100 -> 2, 300 -> 3, >=1000 -> 4
    return pd.cut(mg_per_L, bins=bins, labels=[0, 1, 2, 3, 4], right=False).astype(float)

def egfr_ckdepi_2021(scr, age, female):
    k = np.where(female, 0.7, 0.9); a = np.where(female, -0.241, -0.302)
    return (142 * np.minimum(scr / k, 1) ** a * np.maximum(scr / k, 1) ** -1.200 * 0.9938 ** age
            * np.where(female, 1.012, 1.0))

def yes_no(s):
    return s.map({1.0: 1.0, 2.0: 0.0, 3.0: 0.0})           # 3 = borderline (diabetes) -> no

def build_cycle(period, spec):
    f = {k: read_nhanes(v, period, spec["year"]) for k, v in spec["files"].items()}
    d = f["DEMO"][["SEQN", "RIDAGEYR", "RIAGENDR"]]
    d = d.merge(f["CBC"][["SEQN", "LBXHGB", "LBXHCT", "LBXRBCSI", "LBXWBCSI"]], on="SEQN", how="left")
    d = d.merge(f["BIO"][["SEQN", "LBXSCR", "LBXSBU", "LBXSGL", "LBXSNASI", "LBXSKSI"]], on="SEQN", how="left")
    d = d.merge(f["ALB"][["SEQN", "URXUMA"]], on="SEQN", how="left")
    bp = f["BP"]
    dia_cols = [c for c in bp.columns if c.startswith("BPXDI") or c.startswith("BPXODI")]
    bp = bp[["SEQN"] + dia_cols].copy()
    bp["bp"] = bp[dia_cols].where(bp[dia_cols] > 0).mean(axis=1)
    d = d.merge(bp[["SEQN", "bp"]], on="SEQN", how="left")
    d = d.merge(f["BPQ"][["SEQN", "BPQ020"]], on="SEQN", how="left")
    d = d.merge(f["DIQ"][["SEQN", "DIQ010"]], on="SEQN", how="left")
    d = d.merge(f["MCQ"][["SEQN", "MCQ160C"]], on="SEQN", how="left")
    out = pd.DataFrame({
        "SEQN": d["SEQN"], "cycle": period, "age": d["RIDAGEYR"], "female": d["RIAGENDR"] == 2,
        "hemo": d["LBXHGB"], "pcv": d["LBXHCT"], "rc": d["LBXRBCSI"], "wc": d["LBXWBCSI"] * 1000,
        "bgr": d["LBXSGL"], "bu": d["LBXSBU"] * 2.14,          # BUN (mg/dL) -> urea (mg/dL)
        "sc": d["LBXSCR"], "sod": d["LBXSNASI"], "pot": d["LBXSKSI"],
        "al": albumin_grade(d["URXUMA"]), "bp": d["bp"],
        "htn": yes_no(d["BPQ020"]), "dm": yes_no(d["DIQ010"]), "cad": yes_no(d["MCQ160C"])})
    return out

frames = []
for period, spec in CYCLES.items():
    print("  NHANES", period, "...")
    frames.append(build_cycle(period, spec))
nh = pd.concat(frames, ignore_index=True)
n_total = len(nh)
nh = nh[nh["age"] >= 18]
n_adult = len(nh)
nh = nh.dropna(subset=EXT_FEATURES + ["age"]).reset_index(drop=True)
nh["egfr"] = egfr_ckdepi_2021(nh["sc"].values, nh["age"].values, nh["female"].values)
nh["ckd"] = (nh["egfr"] < 60).astype(int)
X_ext, y_ext = nh[EXT_FEATURES].astype(float), nh["ckd"].values
print(f"NHANES participants: {n_total}; adults: {n_adult}; complete cases: {len(nh)}; "
      f"CKD (eGFR<60): {y_ext.sum()} ({100*y_ext.mean():.1f}%)")
LOG["nhanes"] = dict(participants=int(n_total), adults=int(n_adult), complete_cases=int(len(nh)),
                     ckd=int(y_ext.sum()), notckd=int((1 - y_ext).sum()), prevalence_pct=round(100 * y_ext.mean(), 1))
nh.to_csv(os.path.join(OUT, "supplementary", "S_nhanes_analytic_cohort.csv"), index=False)

# Table 1: dataset summary
table1 = pd.DataFrame({
    "Attribute": ["Repository", "Total instances", "Features used", "CKD / not-CKD", "Missing values",
                  "Geographic origin", "CKD definition"],
    "UCI CKD dataset": ["UCI ML Repository (ID 336)", len(y_all), f"{len(FEATURES)}",
                        f"{n_ckd} / {n_not} ({n_ckd/n_not:.2f}:1)", f"up to {LOG['uci']['max_missing_pct']}% per feature",
                        "Single hospital, Tamil Nadu, India", "Clinical diagnosis label"],
    "NHANES validation cohort": ["CDC/NCHS NHANES 2015-2016 and 2017-March 2020", len(y_ext), f"{len(EXT_FEATURES)} shared",
                                 f"{int(y_ext.sum())} / {int((1-y_ext).sum())} ({y_ext.sum()/(1-y_ext).sum():.2f}:1)",
                                 "complete cases only", "United States, nationally representative survey",
                                 "eGFR < 60 mL/min/1.73 m2 (CKD-EPI 2021)"]})
save_table(table1, "Table1_datasets.csv")

# Feature distribution shift between cohorts
shift = []
for c in EXT_FEATURES:
    u, n_ = X_all[c].dropna(), X_ext[c]
    pooled = np.sqrt((u.var() + n_.var()) / 2)
    shift.append(dict(feature=c, UCI_mean=u.mean(), UCI_sd=u.std(), NHANES_mean=n_.mean(), NHANES_sd=n_.std(),
                      std_mean_diff=(n_.mean() - u.mean()) / pooled if pooled > 0 else np.nan,
                      UCI_missing_pct=100 * X_all[c].isna().mean()))
shift_df = pd.DataFrame(shift).round(3)
save_table(shift_df, "supplementary/S_feature_shift_UCI_vs_NHANES.csv")


## 13. External validation
Each model is re-fitted on the full UCI cohort restricted to the 14 shared features, using the tuned hyperparameters. Imputer, scaler and decision threshold are fitted on UCI only, and the models are then applied to NHANES **without any re-training** (**Table 5**).

Because the NHANES outcome is computed from serum creatinine, a sensitivity analysis repeats the validation without creatinine and with the three complete-blood-count features alone.

In [ ]:
X_all14 = X_all[EXT_FEATURES]
EXT, ext_rows = {}, []
for name in ALL_NAMES:
    params = BEST_PARAMS[name]
    oof = np.zeros(len(y_all)); aucs = []
    for tr, va in CV_SPLITS:
        pipe = fit_pipe(make_pipe(name, EXT_FEATURES, params), name, X_all14.iloc[tr], y_all[tr])
        oof[va] = proba(pipe, X_all14.iloc[va]); aucs.append(roc_auc_score(y_all[va], oof[va]))
    thr = best_f1_threshold(y_all, oof)
    pipe = fit_pipe(make_pipe(name, EXT_FEATURES, params), name, X_all14, y_all)
    p = proba(pipe, X_ext)
    auc, lo, hi = delong_ci(y_ext, p)
    m = metrics_at(y_ext, p, thr)
    EXT[name] = dict(uci14_cv=np.mean(aucs), uci14_sd=np.std(aucs, ddof=1), auc=auc, lo=lo, hi=hi, prob=p)
    ext_rows.append({"Model": name, "Type": "ML" if name in ML_NAMES else "DL",
                     "UCI CV-AUC (14 features)": f"{np.mean(aucs):.3f} ± {np.std(aucs, ddof=1):.3f}",
                     "NHANES AUC": round(auc, 3), "95% CI (DeLong)": f"{lo:.3f}-{hi:.3f}",
                     "Sensitivity": round(m["Sensitivity"], 3), "Specificity": round(m["Specificity"], 3),
                     "Brier score": round(brier_score_loss(y_ext, p), 4), "UCI threshold": round(thr, 3)})
    print(f"  {name:<20} UCI-14 CV-AUC={np.mean(aucs):.3f}  NHANES AUC={auc:.3f} [{lo:.3f}-{hi:.3f}]")
table5 = pd.DataFrame(ext_rows).sort_values("NHANES AUC", ascending=False)
save_table(table5, "Table5_external_validation.csv")

# Sensitivity analysis: the NHANES outcome is derived from creatinine, so models are also evaluated without 'sc'
sens_rows = []
for tag, feats in [("14 features minus serum creatinine", [f for f in EXT_FEATURES if f != "sc"]),
                   ("CBC only (hemo, pcv, rc)", ["hemo", "pcv", "rc"])]:
    for name in ["Logistic Regression", "Random Forest", "Gradient Boosting"]:
        pipe = fit_pipe(make_pipe(name, feats, BEST_PARAMS[name]), name, X_all[feats], y_all)
        auc, lo, hi = delong_ci(y_ext, proba(pipe, X_ext[feats]))
        sens_rows.append(dict(Feature_set=tag, Model=name, NHANES_AUC=round(auc, 3), CI=f"{lo:.3f}-{hi:.3f}"))
sens_df = pd.DataFrame(sens_rows)
save_table(sens_df, "supplementary/S_external_without_creatinine.csv")


## 14. Figures
Main figures (300 dpi, saved in `figures/`):

| Figure | Content |
|---|---|
| 2 | Confusion matrices of all ten models (hold-out test set) |
| 3 | ROC curves: (a) ML models, (b) DL models |
| 4 | Grouped comparison of accuracy, F1, sensitivity, specificity and ROC-AUC |
| 5 | UCI cross-validated vs NHANES external ROC-AUC with 95% CIs |
| 6 | (a) Feature ablation; (b) XAI consensus ranking heatmap |
| 7 | SHAP: (a) global importance; (b) instance-level beeswarm |
| 8 | Partial dependence with ICE curves for the four top-ranked continuous features |

In [ ]:
C_ML, C_DL = "#2E5A88", "#C0504D"

# Figure 2: confusion matrices
fig, axes = plt.subplots(2, 5, figsize=(16, 6.6))
for ax, name in zip(axes.ravel(), ALL_NAMES):
    h = HELD[name]
    cm = np.array([[h["TN"], h["FP"]], [h["FN"], h["TP"]]])
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.sum())
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14, fontweight="bold",
                    color="white" if cm[i, j] > cm.sum() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["not-CKD", "CKD"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["not-CKD", "CKD"])
    ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
    ax.set_title(f"{name}\nAccuracy = {100*h['Accuracy']:.2f}%")
fig.tight_layout(); save_fig(fig, "Fig2_confusion_matrices.png")

# Figure 3: ROC curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, names, ttl in [(axes[0], ML_NAMES, "(a) Machine learning models"), (axes[1], DL_NAMES, "(b) Deep learning models")]:
    ax.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="Chance (AUC = 0.500)")
    for name in names:
        fpr, tpr, _ = roc_curve(y_te, HELD[name]["prob"])
        ax.plot(fpr, tpr, lw=1.8, label=f"{name} (AUC = {HELD[name]['auc']:.3f})")
    ax.set_xlabel("False positive rate (1 - specificity)"); ax.set_ylabel("True positive rate (sensitivity)")
    ax.set_title(ttl); ax.legend(fontsize=8, loc="lower right")
fig.tight_layout(); save_fig(fig, "Fig3_roc_curves.png")

# Figure 4: grouped bar chart of held-out metrics
mets = ["Accuracy", "F1", "Sensitivity", "Specificity", "auc"]
labels = ["Accuracy", "F1 score", "Sensitivity", "Specificity", "ROC-AUC"]
fig, ax = plt.subplots(figsize=(14, 5))
w = 0.16; x = np.arange(len(ALL_NAMES))
for i, (m, lab) in enumerate(zip(mets, labels)):
    ax.bar(x + (i - 2) * w, [100 * HELD[n][m] for n in ALL_NAMES], w, label=lab)
ax.set_xticks(x); ax.set_xticklabels([SHORT[n] for n in ALL_NAMES])
lo_y = min(100 * HELD[n][m] for n in ALL_NAMES for m in mets)
ax.set_ylim(max(0, np.floor(lo_y / 5) * 5 - 5), 101)
ax.set_xlabel("Model"); ax.set_ylabel("Score (%)")
ax.axvline(4.5, color="grey", ls=":", lw=1)
ax.legend(ncol=5, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(); save_fig(fig, "Fig4_performance_comparison.png")

# Figure 5: UCI (14 shared features, 5-fold CV) vs NHANES external AUC
fig, ax = plt.subplots(figsize=(13, 5))
u = [EXT[n]["uci14_cv"] for n in ALL_NAMES]; e = [EXT[n]["auc"] for n in ALL_NAMES]
err = np.array([[EXT[n]["auc"] - EXT[n]["lo"] for n in ALL_NAMES], [EXT[n]["hi"] - EXT[n]["auc"] for n in ALL_NAMES]])
ax.bar(x - 0.2, u, 0.4, label="UCI, 5-fold CV (14 shared features)", color=C_ML)
ax.bar(x + 0.2, e, 0.4, yerr=err, capsize=3, label="NHANES external validation (95% CI)", color="#8CC084")
ax.axhline(0.9, color="#D9822B", ls="--", lw=1, label="AUC = 0.90")
ax.set_xticks(x); ax.set_xticklabels([SHORT[n] for n in ALL_NAMES])
ax.set_ylim(max(0, min(min(u), min(EXT[n]["lo"] for n in ALL_NAMES)) - 0.05), 1.01)
ax.set_xlabel("Model"); ax.set_ylabel("ROC-AUC"); ax.legend(fontsize=8, loc="lower right")
fig.tight_layout(); save_fig(fig, "Fig5_external_validation.png")

# Figure 6: (a) ablation  (b) consensus heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]; keys = list(ABL.keys()); xx = np.arange(len(keys))
aucm = np.array([ABL[k]["auc"] for k in keys]); sd = np.array([ABL[k]["sd"] for k in keys])
ax.fill_between(xx, aucm - sd, np.minimum(aucm + sd, 1), alpha=0.2, color=C_ML, label="CV-AUC ± 1 SD")
ax.plot(xx, aucm, "o-", color=C_ML, label="CV-AUC")
ax.plot(xx, [ABL[k]["sens"] for k in keys], "s--", color="#E46C0A", label="Sensitivity")
ax.plot(xx, [ABL[k]["spec"] for k in keys], "^--", color="#4F8A3C", label="Specificity")
ax.set_xticks(xx); ax.set_xticklabels(keys)
ax.set_xlabel("Feature subset (XAI consensus order)"); ax.set_ylabel("Score")
ax.set_title("(a) Feature ablation (Random Forest, 5-fold CV)"); ax.legend(fontsize=8, loc="lower right")
ax = axes[1]
hm = rank_df.iloc[:12][["SHAP-RF", "SHAP-MLP", "Permutation", "Gini (RF)"]]
sns.heatmap(hm, annot=True, fmt=".0f", cmap="Blues_r", ax=ax, cbar_kws={"label": "Rank (1 = most important)"})
ax.set_xlabel("XAI attribution method"); ax.set_ylabel("Feature")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
ax.set_title(f"(b) XAI consensus ranking (Kendall's W = {W_all:.3f})")
fig.tight_layout(); save_fig(fig, "Fig6_ablation_consensus.png")

# Figure 7: SHAP (a) global bar (b) beeswarm
order = np.argsort(shap_rf)[::-1][:15]
fig = plt.figure(figsize=(15, 6))
ax1 = fig.add_subplot(1, 2, 1)
ax1.barh([FEATURES[i] for i in order][::-1], shap_rf[order][::-1], color=C_ML)
ax1.set_xlabel("Mean |SHAP value| (impact on P(CKD))"); ax1.set_ylabel("Feature")
ax1.set_title("(a) Global feature importance (SHAP, Random Forest)")
ax2 = fig.add_subplot(1, 2, 2); plt.sca(ax2)
shap.summary_plot(sv_rf, Xte_s, feature_names=FEATURES, max_display=15, show=False, plot_size=None)
ax2 = plt.gca()
ax2.set_xlabel("SHAP value (positive = towards CKD)"); ax2.set_ylabel("Feature")
ax2.set_title("(b) Instance-level SHAP attributions (beeswarm)")
fig.tight_layout(); save_fig(fig, "Fig7_shap.png")

# Figure 8: PDP + ICE for top-4 continuous consensus features (original clinical units)
UNITS = {"hemo": "Hemoglobin (g/dL)", "pcv": "Packed cell volume (%)", "rc": "Red cell count (millions/cmm)",
         "sc": "Serum creatinine (mg/dL)", "bu": "Blood urea (mg/dL)", "bgr": "Blood glucose (mg/dL)",
         "sg": "Urine specific gravity", "al": "Albumin grade (0-5)", "sod": "Sodium (mEq/L)",
         "pot": "Potassium (mEq/L)", "wc": "White cell count (cells/cmm)", "bp": "Diastolic BP (mm Hg)",
         "age": "Age (years)", "su": "Sugar grade (0-5)"}
cont = [f for f in CONSENSUS if f in UNITS][:4]
X_pdp = pd.DataFrame(rf_pipe.named_steps["imp"].transform(X_tr), columns=FEATURES)
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
PartialDependenceDisplay.from_estimator(rf_pipe, X_pdp, cont, kind="both", subsample=min(100, len(X_pdp)),
                                        random_state=SEED, ax=axes.ravel(),
                                        ice_lines_kw={"alpha": 0.15, "color": "#7A9CC6"},
                                        pd_line_kw={"color": "#B22222", "lw": 2.5})
for ax, f, lab in zip(fig.axes[-4:], cont, "abcd"):
    ax.set_xlabel(UNITS[f]); ax.set_ylabel("Predicted P(CKD)")
fig.tight_layout(); save_fig(fig, "Fig8_pdp_ice.png")
LOG["pdp_features"] = cont

### 14.1 Supplementary figures
LIME explanations, permutation importance for all ML models, the PCA decision boundary, TabNet attention masks and SHAP for the MLP.

In [ ]:
lime_exp = lime.lime_tabular.LimeTabularExplainer(X_pdp.values, feature_names=FEATURES,
                                                  class_names=["not-CKD", "CKD"], discretize_continuous=True,
                                                  random_state=SEED)
X_te_imp = pd.DataFrame(rf_pipe.named_steps["imp"].transform(X_te), columns=FEATURES)
idx_ckd = int(np.where(y_te == 1)[0][0]); idx_not = int(np.where(y_te == 0)[0][0])
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for r, name in enumerate(["Random Forest", "MLP"]):
    fn = lambda A, n=name: FITTED[n].predict_proba(pd.DataFrame(A, columns=FEATURES))
    for c, (idx, lab) in enumerate([(idx_ckd, "CKD patient"), (idx_not, "not-CKD patient")]):
        e = lime_exp.explain_instance(X_te_imp.values[idx], fn, num_features=10)
        lst = e.as_list()[::-1]
        axes[r, c].barh([a for a, _ in lst], [b for _, b in lst],
                        color=[C_DL if b > 0 else C_ML for _, b in lst])
        axes[r, c].set_xlabel("LIME weight (positive = towards CKD)"); axes[r, c].set_ylabel("Feature condition")
        axes[r, c].set_title(f"{name}: {lab}")
fig.tight_layout(); save_fig(fig, "S_lime.png", sub=True)

# Permutation importance for all ML models
fig, ax = plt.subplots(figsize=(12, 6))
pi = pd.DataFrame(PERM, index=FEATURES).loc[CONSENSUS[:12]]
pi.plot.barh(ax=ax); ax.invert_yaxis()
ax.set_xlabel("Mean decrease in ROC-AUC"); ax.set_ylabel("Feature"); ax.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "S_permutation_importance.png", sub=True)

# 2-D PCA decision boundary
pca = PCA(2, random_state=SEED).fit(Xtr_s)
Ztr, Zte = pca.transform(Xtr_s), pca.transform(Xte_s)
gx, gy = np.meshgrid(np.linspace(Ztr[:, 0].min() - 1, Ztr[:, 0].max() + 1, 300),
                     np.linspace(Ztr[:, 1].min() - 1, Ztr[:, 1].max() + 1, 300))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, name in zip(axes, ["Random Forest", "Gradient Boosting", "SVM (RBF)"]):
    est = clone(FITTED[name].named_steps["clf"])
    if name in SAMPLE_WEIGHT_MODELS:
        est.fit(Ztr, y_tr, sample_weight=compute_sample_weight("balanced", y_tr))
    else:
        est.fit(Ztr, y_tr)
    zz = est.predict_proba(np.c_[gx.ravel(), gy.ravel()])[:, 1].reshape(gx.shape)
    ax.contourf(gx, gy, zz, levels=20, cmap="RdBu_r", alpha=0.6)
    ax.scatter(Zte[:, 0], Zte[:, 1], c=y_te, cmap="RdBu_r", edgecolor="k", s=25)
    ax.set_xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}% variance)")
    ax.set_ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}% variance)")
    ax.set_title(name)
fig.tight_layout(); save_fig(fig, "S_decision_boundary_pca.png", sub=True)

# TabNet attention mask importance and SHAP-MLP
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
o = np.argsort(tabnet_imp)[::-1][:15]
axes[0].barh([FEATURES[i] for i in o][::-1], tabnet_imp[o][::-1], color="#6C4F9E")
axes[0].set_xlabel("Normalised TabNet attention-mask importance"); axes[0].set_ylabel("Feature")
axes[0].set_title("(a) TabNet built-in feature selection")
o = np.argsort(shap_mlp)[::-1][:15]
axes[1].barh([FEATURES[i] for i in o][::-1], shap_mlp[o][::-1], color=C_DL)
axes[1].set_xlabel("Mean |SHAP value| (logit scale)"); axes[1].set_ylabel("Feature")
axes[1].set_title("(b) SHAP GradientExplainer, MLP")
fig.tight_layout(); save_fig(fig, "S_tabnet_and_shap_mlp.png", sub=True)

## 15. Reproducibility record
Saves the software versions, hardware and random seed, writes all key results to `key_results.json`, prints a summary table and packages the output folder.

In [ ]:
import importlib.metadata as md_
def _v(pkg):
    try: return md_.version(pkg)
    except Exception: return "n/a"
env = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
       "scikit-learn": sklearn.__version__, "torch": torch.__version__, "pytorch-tabnet": _v("pytorch-tabnet"),
       "shap": shap.__version__, "lime": _v("lime"), "statsmodels": _v("statsmodels"), "scipy": _v("scipy"),
       "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
       "seed": SEED, "FAST_mode": FAST}
with open(os.path.join(OUT, "software_versions.txt"), "w") as f:
    for k, v in env.items():
        f.write(f"{k}: {v}\n")

best_held = max(ALL_NAMES, key=lambda n: (HELD[n]["auc"], HELD[n]["Accuracy"]))
best_ext = max(ALL_NAMES, key=lambda n: EXT[n]["auc"])
LOG["heldout"] = {n: dict(acc=round(100 * HELD[n]["Accuracy"], 2), f1=round(HELD[n]["F1"], 3),
                          sens=round(HELD[n]["Sensitivity"], 3), spec=round(HELD[n]["Specificity"], 3),
                          auc=round(HELD[n]["auc"], 3), cv_auc=f"{CV[n]['auc_mean']:.3f} ± {CV[n]['auc_sd']:.3f}")
                  for n in ALL_NAMES}
LOG["external"] = {n: dict(auc=round(EXT[n]["auc"], 3), ci=f"{EXT[n]['lo']:.3f}-{EXT[n]['hi']:.3f}") for n in ALL_NAMES}
LOG["ablation"] = {k: dict(auc=round(v["auc"], 3), sd=round(v["sd"], 3), sens=round(v["sens"], 3),
                           spec=round(v["spec"], 3)) for k, v in ABL.items()}
LOG["best_heldout_model"], LOG["best_external_model"] = best_held, best_ext
LOG["environment"] = env
with open(os.path.join(OUT, "key_results.json"), "w") as f:
    json.dump(LOG, f, indent=2, default=str)

zpath = os.path.join(os.path.dirname(OUT.rstrip("/")) or ".", "CKD_XAI_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files:
            full = os.path.join(root, fn)
            if "nhanes_raw" not in full:
                z.write(full, os.path.relpath(full, OUT))
print("\nSoftware:", env)
summary = pd.DataFrame({n: {"Held-out accuracy (%)": LOG["heldout"][n]["acc"], "Held-out AUC": LOG["heldout"][n]["auc"],
                            "5-fold CV-AUC": LOG["heldout"][n]["cv_auc"], "NHANES AUC": LOG["external"][n]["auc"],
                            "NHANES 95% CI": LOG["external"][n]["ci"]} for n in ALL_NAMES}).T
display(Markdown("### Summary of results")); display(summary)
print(f"Best held-out model: {best_held} | best external model: {best_ext}")
print(f"\nAll results saved in {OUT}\nZip file: {zpath}")
try:
    from google.colab import files
    files.download(zpath)
except Exception:
    pass

## Output structure

```
CKD_XAI_results/
├── tables/          Table1–Table7 (CSV), Kendall's W
├── figures/         Fig2–Fig8 (PNG, 300 dpi)
├── supplementary/   hyperparameters, imputation sensitivity, missingness, feature shift,
│                    external validation without creatinine, NHANES analytic cohort, extra XAI figures
├── key_results.json
└── software_versions.txt
```

**Data sources**: UCI Machine Learning Repository, Chronic Kidney Disease dataset (doi:10.24432/C5G020); Centers for Disease Control and Prevention, National Health and Nutrition Examination Survey (https://wwwn.cdc.gov/nchs/nhanes/).